# 6-1 순환 신경망

본 노트북은 본문 6-1 절의 코드 예제([코드 6-1] ~ [코드 6-3])를 절 흐름 그대로 모은 것이다. 순차 데이터를 순환 신경망 입력으로 다듬는 두 단계(어휘 사전과 인코딩, 슬라이딩 윈도우)를 직접 구현한다.

다루는 내용
- `[코드 6-1]` 어휘 사전과 원-핫 인코딩
- `[코드 6-2]` '도도솔솔라라솔'을 같은 길이의 데이터 샘플로 재구성
- `[코드 6-3]` `<eos>`와 `<pad>`를 추가한 데이터 샘플

## 참고 - 라이브러리 설치 (이미 설치된 경우 건너뛰어도 좋다)

torch 는 기본 의존성으로 README 에서 안내한다. 별도 설치가 필요하면 아래를 사용한다.

In [1]:
# 참고 - 라이브러리 설치 (이미 설치된 경우 건너뛰어도 좋다)
# !pip install -q torch

## 라이브러리 import 와 시드 고정

In [1]:
# 참고 - 환경 설정 (code_reference 모듈 임포트 경로)
import sys
sys.path.append('../../')

from code_reference import common

In [2]:
# 참고 - 라이브러리 import와 시드 고정
import random
import torch
import numpy as np
    
# 시드 고정
common.set_seed()
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

42

## [코드 6-1] 어휘 사전과 원-핫 인코딩

순차 데이터를 모델에 넣으려면 각 요소를 크기가 의미를 가지지 않는 수로 바꿔야 한다. 일곱 음계를 토큰으로 가지는 어휘 사전(`vocab`)을 딕셔너리로 만든 뒤, `'도도솔솔'`을 고유 번호 텐서로 바꾸고 `F.one_hot()` 으로 원-핫 인코딩한다. 모델 입력으로 쓰려면 `.float()` 으로 실수형 텐서로 변환한다.

In [3]:
###############################################################################
# 코드 6-1 - 어휘 사전과 원-핫 인코딩
###############################################################################
import torch
import torch.nn.functional as F

# 어휘 사전: 보통 고유 번호는 0부터 차례대로 부여 (관례상 vocab 변수명 사용)
# 토큰에 따른 번호의 순서는 의미가 없으므로 가나다순으로 바꿔 매겨도 무방
vocab = {'도': 0, '레': 1, '미': 2, '파': 3, '솔': 4, '라': 5, '시': 6}

# '도도솔솔'의 원-핫 인코딩 과정
sequence = '도도솔솔'
# ① 음을 어휘 사전의 고유 번호 텐서로 변환
sequence_idx = torch.tensor([vocab[token] for token in sequence])
print('고유 번호의 텐서:', sequence_idx)

# ② 3장에서 소개한 F.one_hot()으로 원-핫 인코딩
vocab_size = len(vocab)                # 어휘 사전의 크기 = 고유한 요소의 개수
# .float(): 모델의 학습 데이터로 사용하려면 실수형 텐서로 변환해야 함
sequence_onehot = F.one_hot(sequence_idx, num_classes=vocab_size).float()
print('원-핫 인코딩된 텐서:')
print(sequence_onehot)

고유 번호의 텐서: tensor([0, 0, 4, 4])
원-핫 인코딩된 텐서:
tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.]])


결과를 보면 '도도솔솔'의 네 토큰이 각각 길이 7(= 어휘 사전 크기)의 원-핫 벡터로 변환된다. 0번 토큰('도')은 첫 자리가, 4번 토큰('솔')은 다섯 번째 자리가 1이다.

## [코드 6-2] '도도솔솔라라솔'을 같은 길이의 데이터 샘플로 재구성

슬라이딩 윈도우 기법으로 순차 데이터를 입력과 정답 쌍의 학습 샘플로 재구성한다. 크기 5인 윈도우를 한 칸씩 옮겨 가며, 윈도우 안 5개 요소 중 앞 4개는 입력으로, 마지막 1개는 정답으로 사용한다.

In [4]:
###############################################################################
# 코드 6-2 - '도도솔솔라라솔'을 같은 길이의 데이터 샘플로 재구성
###############################################################################

sequence = '도도솔솔라라솔'
# 이전 음 4개로 다음 음을 예측하기 위해 크기 5인 윈도우를 사용
window_size = 5                                        # 윈도우의 크기
dataset = []
for i in range(len(sequence) - window_size + 1):
    subsequence = sequence[i: i + window_size]
    # 입력: 마지막을 뺀 앞 요소들, 정답: 마지막 1개 요소
    dataset.append((subsequence[:-1], subsequence[-1]))

print('입력 데이터 -> 정답')
for input_data, label in dataset:
    print(f'{input_data}    -> {label}')

입력 데이터 -> 정답
도도솔솔    -> 라
도솔솔라    -> 라
솔솔라라    -> 솔


윈도우가 '도도솔솔라라솔'의 마지막 '솔'까지 도달했지만, 노래가 여기서 끝난다는 정보는 아직 데이터에 담겨 있지 않다. 이 빈틈을 특수 토큰으로 메운다.

## [코드 6-3] `<eos>`와 `<pad>`를 추가한 데이터 샘플

순차 데이터의 끝을 표시하는 종료 토큰 `<eos>` 를 덧붙인 뒤 슬라이딩 윈도우로 샘플을 만든다. 윈도우가 끝까지 밀리며 길이가 모자라는 자리는 패딩 토큰 `<pad>` 로 채운다. 원-핫 인코딩까지 하려면 두 특수 토큰도 어휘 사전에 등록해야 한다.

In [5]:
###############################################################################
# 코드 6-3 - <eos>와 <pad>를 추가한 데이터 샘플
###############################################################################

samples = []
# 순차 데이터에 <eos> 추가
# 길이와 상관없이 특수 토큰 전체가 하나의 토큰으로 취급되도룩 추가
sequence_list = list(sequence)
sequence_list.append('<eos>')
# 순차 데이터의 끝부분까지 포함하도록 윈도우를 끝까지 밀어 가며 샘플 생성
for i in range(len(sequence_list) - 1):
    subsequence = sequence_list[i: i + window_size]
    # 모든 샘플의 길이가 같아지도록 부족한 만큼 <pad>로 채움
    while len(subsequence) < window_size:
        subsequence.append('<pad>')
    samples.append((subsequence[:4], subsequence[-1]))

# 추후 원-핫 인코딩을 해야 한다면 어휘 사전에 특수 토큰도 추가해야 함
last_index = len(vocab)
vocab['<eos>'] = last_index             # <eos> 토큰 추가
vocab['<pad>'] = last_index + 1         # <pad> 토큰 추가

print('입력 데이터 -> 정답')
for input_data, label in samples:
    print(f'{input_data} -> {label}')

입력 데이터 -> 정답
['도', '도', '솔', '솔'] -> 라
['도', '솔', '솔', '라'] -> 라
['솔', '솔', '라', '라'] -> 솔
['솔', '라', '라', '솔'] -> <eos>
['라', '라', '솔', '<eos>'] -> <pad>
['라', '솔', '<eos>', '<pad>'] -> <pad>
['솔', '<eos>', '<pad>', '<pad>'] -> <pad>


본문 설명대로 `<eos>` 다음에 `<pad>` 가 줄줄이 따라오는 형태는 실제 학습에서 잘 쓰지 않는다. `<eos>` 이후의 정보는 무의미하기 때문이며, 위 예제는 `<pad>` 의 개념과 추가 방법을 보여주기 위한 것이다.

In [6]:
# 참고 - 특수 토큰이 추가된 어휘 사전 확인

print('확장된 어휘 사전:', vocab)
print(f'어휘 사전의 크기: {len(vocab)}')

확장된 어휘 사전: {'도': 0, '레': 1, '미': 2, '파': 3, '솔': 4, '라': 5, '시': 6, '<eos>': 7, '<pad>': 8}
어휘 사전의 크기: 9


## 정리

- 순차 데이터의 각 요소는 어휘 사전으로 고유 번호를 부여한 뒤 원-핫 인코딩해 값의 크기 편견을 없앤다.
- 슬라이딩 윈도우로 일정한 길이의 입력·정답 쌍 샘플을 만들면 길이가 제각각인 순차 데이터를 미니배치로 묶을 수 있다.
- `<eos>` 로 시퀀스의 끝을, `<pad>` 로 길이 정렬용 여백을 표시하며, 사용할 특수 토큰은 어휘 사전에도 함께 등록한다.

In [8]:
###############################################################################
# 코드 6-4 김소월의 시, '엄마야 누나야' (연습 문제 6-1)
###############################################################################

poem = '엄마야 누나야 강변 살자 ' \
       '뜰에는 반짝이는 금모래 빛 ' \
       '뒷문 밖에는 갈잎의 노래 ' \
       '엄마야 누나야 강변 살자'

print(poem)

엄마야 누나야 강변 살자 뜰에는 반짝이는 금모래 빛 뒷문 밖에는 갈잎의 노래 엄마야 누나야 강변 살자
